In [ ]:
import os

os.makedirs("data", exist_ok=True)
os.makedirs("models", exist_ok=True)

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving StudentPerformanceFactors.csv to StudentPerformanceFactors.csv
Saving CollegePlacement.csv to CollegePlacement.csv


In [ ]:
import pandas as pd

perf = pd.read_csv("StudentPerformanceFactors.csv")
place = pd.read_csv("CollegePlacement.csv")

print(perf.head())
print(place.head())

   Hours_Studied  Attendance  Sleep_Hours  Previous_Scores Internet_Access  \
0             23          84            7               73             Yes   
1             19          64            8               59             Yes   
2             24          98            7               91             Yes   
3             29          89            8               98             Yes   
4             19          92            6               65             Yes   

   Exam_Score  
0          67  
1          61  
2          74  
3          71  
4          70  
  College_ID  Prev_Sem_Result  CGPA  Academic_Performance  \
0    CLG0030             6.61  6.28                     8   
1    CLG0061             5.52  5.37                     8   
2    CLG0036             5.36  5.83                     9   
3    CLG0055             5.47  5.75                     6   
4    CLG0004             7.91  7.69                     7   

  Internship_Experience  Extra_Curricular_Score  Communication_Skill

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

# Load data
perf = pd.read_csv("StudentPerformanceFactors.csv")

# Convert Internet Access
perf['Internet_Access'] = perf['Internet_Access'].map({'Yes': 1, 'No': 0})

# Calculate median Exam_Score to use as a dynamic threshold
median_exam_score = perf['Exam_Score'].median()

# Create target based on the median score to ensure at least two classes
perf['Result'] = perf['Exam_Score'].apply(lambda x: 1 if x >= median_exam_score else 0)

# Features
X = perf[['Hours_Studied', 'Attendance', 'Sleep_Hours', 'Previous_Scores', 'Internet_Access']]
y = perf['Result']

# Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42) # Added random_state for reproducibility

# Train model
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

print("Accuracy:", model.score(X_test, y_test))


Accuracy: 0.8638426626323752


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import pandas as pd

place = pd.read_csv("CollegePlacement.csv")

place['Internship_Experience'] = place['Internship_Experience'].astype(str).str.strip().str.lower()
place['Placement'] = place['Placement'].astype(str).str.strip().str.lower()

place['Internship_Experience'] = place['Internship_Experience'].map({'yes': 1, 'no': 0})
place['Placement'] = place['Placement'].map({'yes': 1, 'no': 0})

numeric_cols = ['CGPA', 'Communication_Skills', 'Projects_Completed', 'Extra_Curricular_Score']
for col in numeric_cols:
    place[col] = place[col].fillna(place[col].median())

place = place.dropna(subset=['Internship_Experience', 'Placement'])

features = [
    'CGPA',
    'Internship_Experience',
    'Communication_Skills',
    'Projects_Completed',
    'Extra_Curricular_Score'
]

target = 'Placement'

X = place[features]
y = place[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42)

model_place = RandomForestClassifier(
    class_weight={0:1, 1:1.5},
    n_estimators=100,
    max_depth=5,
    min_samples_split=10,
    random_state=42
)

model_place.fit(X_train, y_train)
y_pred = model_place.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print("Placement Model Accuracy:", accuracy)

Placement Model Accuracy: 0.9208


In [ ]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.91      1.00      0.95      2096
           1       1.00      0.51      0.68       404

    accuracy                           0.92      2500
   macro avg       0.96      0.75      0.82      2500
weighted avg       0.93      0.92      0.91      2500



In [ ]:
import pickle

with open("models/performance_model.pkl", "wb") as f:
    pickle.dump(model, f)   # your logistic regression model

# Save Placement Model
with open("models/placement_model.pkl", "wb") as f:
    pickle.dump(model_place, f)